# 75 — Campaign C staged M4（child M3 完走後）

前提: notebook 74 で child M3 が `M3_COMPLETE` / `CORE_REPRODUCED`。

- **CUDA GPU 必須**（`require_cuda=True`）
- 親は **今回の child M3**（j2_max=2）。デフォルト j2=1 親は使わない
- M4 完了状態は **`BLOCKED_MATH` / `NOT_CERTIFIED`**（決定的 RSVD/GPU/cutoff 境界なし）
- M5 は自動開始しない。influence proxy / residual は探索値のみ


In [ ]:
from pathlib import Path
import os
import sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm4_orchestrator.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m4_orchestrator.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault(
    'VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT',
)
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
os.environ.setdefault('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('M4 child lineage requires CUDA GPU.')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
import json
from pathlib import Path

from src.common import read_json

M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
DEFAULT_CAND = os.environ.get('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')
explicit_pkg = os.environ.get('VALIDATED_RG_STAGED_PACKAGE')
PACKAGE_ROOT = (
    Path(explicit_pkg).expanduser().resolve()
    if explicit_pkg else
    (PERSIST_ROOT / 'searches' / M7C_RUN_ID / 'auto_execute' / DEFAULT_CAND).resolve()
)
required = [
    'scheme.json', 'resource_gate.json', 'child_run_ids.json',
    'm4_config_overrides.json',
]
missing = [n for n in required if not (PACKAGE_ROOT / n).is_file()]
if missing:
    raise FileNotFoundError(f'Package incomplete: {PACKAGE_ROOT} missing {missing}')

CHILD_IDS = read_json(PACKAGE_ROOT / 'child_run_ids.json')
M3_RUN_ID = CHILD_IDS['M3']
M4_RUN_ID = CHILD_IDS['M4']
os.environ['VALIDATED_RG_M3_RUN_ID'] = M3_RUN_ID
os.environ['VALIDATED_RG_M4_RUN_ID'] = M4_RUN_ID

m3_acceptance = PERSIST_ROOT / 'runs' / M3_RUN_ID / 'reports' / 'M3_acceptance.json'
m3_report = PERSIST_ROOT / 'runs' / M3_RUN_ID / 'reports' / 'M3_report.json'
if not m3_acceptance.is_file() or not m3_report.is_file():
    raise FileNotFoundError(
        f'Child M3 not complete: missing acceptance/report under {M3_RUN_ID}. '
        'Finish notebook 74 first.'
    )
print('PACKAGE_ROOT', PACKAGE_ROOT)
print('M3_RUN_ID', M3_RUN_ID)
print('M4_RUN_ID', M4_RUN_ID)


## M4Config（child M3 親 + package overrides）


In [ ]:
from dataclasses import asdict
from pathlib import Path

from src.m4_config import M4Config
from src.m4_parent import verify_accepted_m3_parent
from src.m7_lineage import effective_projected_rank
from src.m7_staged_lineage import write_child_m3_acceptance_audit

over = read_json(PACKAGE_ROOT / 'm4_config_overrides.json')
m3_report_doc = read_json(PERSIST_ROOT / 'runs' / M3_RUN_ID / 'reports' / 'M3_report.json')
rsvd = (m3_report_doc.get('results') or {}).get('M3_RSVD', {}).get('result', {})
m3_target_rank = int(rsvd.get('target_rank') or 0)
if m3_target_rank < 1:
    raise RuntimeError('Child M3 RSVD target_rank missing.')
expected_projected = effective_projected_rank(m3_target_rank)
package_projected = int(over['projected_rank'])
if package_projected != expected_projected:
    print(
        'WARNING: m4_config_overrides.projected_rank', package_projected,
        '!= effective_projected_rank(M3 target_rank=', m3_target_rank, ') =',
        expected_projected, '; using effective value from child M3.',
    )
projected_rank = expected_projected
operator_dimension = int(over['operator_dimension'])
m3_op_dim = int(
    (m3_report_doc.get('config') or {}).get('operator_dimension')
    or rsvd.get('dimension')
    or 0
)
if m3_op_dim and m3_op_dim != operator_dimension:
    raise RuntimeError(
        f'operator_dimension override {operator_dimension} != child M3 {m3_op_dim}'
    )

audit_path = PROJECT_ROOT / 'audit' / 'm3_accepted_parent.json'
audit = read_json(audit_path) if audit_path.is_file() else None
if not isinstance(audit, dict) or audit.get('accepted_run_id') != M3_RUN_ID:
    print(
        'Rewriting audit/m3_accepted_parent.json from child M3:',
        M3_RUN_ID,
        '(was', None if not isinstance(audit, dict) else audit.get('accepted_run_id'), ')',
    )
    audit = write_child_m3_acceptance_audit(
        PROJECT_ROOT,
        run_root=PERSIST_ROOT / 'runs' / M3_RUN_ID,
    )
if not isinstance(audit, dict):
    raise RuntimeError(f'Malformed audit: {audit_path}')
if audit.get('accepted_run_id') != M3_RUN_ID:
    raise RuntimeError(
        f"audit accepted_run_id={audit.get('accepted_run_id')!r} != package M3={M3_RUN_ID!r} "
        'after rewrite.'
    )

base = asdict(M4Config())
base.update({
    'parent_run_id': audit['accepted_run_id'],
    'parent_checkpoint': Path(audit['checkpoint_path']).name,
    'parent_checkpoint_path': audit['checkpoint_path'],
    'parent_report_path': audit['m3_report_path'],
    'parent_acceptance_path': audit['m3_acceptance_path'],
    'operator_dimension': operator_dimension,
    'projected_rank': projected_rank,
    'require_cuda': True,
})
M4_CONFIG = M4Config(**base)
M3_EVIDENCE = verify_accepted_m3_parent(PROJECT_ROOT, M4_CONFIG)
print(json.dumps({
    'parent_run': M4_CONFIG.parent_run_id,
    'parent_checkpoint': M4_CONFIG.parent_checkpoint,
    'checkpoint_index': audit.get('checkpoint_index'),
    'operator_dimension': M4_CONFIG.operator_dimension,
    'projected_rank': M4_CONFIG.projected_rank,
    'm3_target_rank': m3_target_rank,
    'parent_tensor_shapes': {k: list(v.shape) for k, v in M3_EVIDENCE.tensors.items()},
    'screening': M3_EVIDENCE.metrics.get('screening'),
    'M4_target_status': M4_CONFIG.milestone_status,
    'certification_status': M4_CONFIG.certification_status,
}, indent=2, ensure_ascii=False, default=str))


## テスト報告

再開で時間が惜しいときは `SKIP_M4_TESTS=1`（既存 `m4_test_report.json` を再利用）。


In [ ]:
import time
import pytest
from src.common import atomic_write_json, read_json

saved = PACKAGE_ROOT / 'm4_test_report.json'
skip = os.environ.get('SKIP_M4_TESTS', '').strip() in {'1', 'true', 'TRUE', 'yes'}
if skip and saved.is_file():
    M4_TEST_REPORT = read_json(saved)
    print('Reusing saved m4_test_report.json')
else:
    started = time.monotonic()
    os.chdir(PROJECT_ROOT)
    cpu_rc = pytest.main([
        '-q', f'--rootdir={PROJECT_ROOT}', '-m', 'not gpu',
        'tests/test_forward_ad.py',
        'tests/test_m4_restart.py',
        'tests/test_matrix_free.py',
        'tests/test_m3_restart.py',
    ])
    if cpu_rc != 0:
        raise RuntimeError(f'M4-related CPU suite failed: {cpu_rc}')
    gpu_rc = pytest.main([
        '-q', f'--rootdir={PROJECT_ROOT}',
        'tests/test_forward_ad.py', 'tests/test_m4_restart.py', '-m', 'gpu',
    ])
    if gpu_rc != 0:
        raise RuntimeError(f'M4 required GPU suite failed: {gpu_rc}')
    M4_TEST_REPORT = {
        'accepted_m3_parent': 'PASS',
        'm0_m1_m2_m3_regression_cpu_suite': 'PASS',
        'm4_required_cpu_suite': 'PASS',
        'm4_required_gpu_suite': 'PASS',
        'm4_fresh_process_resume': 'PASS',
        'm4_derivative_checkpoint_restore': 'PASS',
        'elapsed_s': time.monotonic() - started,
        'suite': 'staged_m4_notebook',
    }
    atomic_write_json(saved, M4_TEST_REPORT)
print(json.dumps(M4_TEST_REPORT, indent=2))


## M4 セッション実行（checkpoint まで）

未完了なら **同じセルを再実行**（同じ `M4_RUN_ID` で resume）。
初回セッションは約2時間 hard-return、再開セッションは約5.5時間政策（orchestrator 側）。


In [ ]:
from src.m4_orchestrator import create_or_resume_m4
from src.runtime import environment_info, validate_persistent_root

PERSIST = validate_persistent_root()
print(json.dumps(environment_info(), ensure_ascii=False, indent=2, default=str))

os.environ.setdefault('VALIDATED_RG_M4_ALLOW_CODE_DRIFT', '1')
m4_orchestrator = create_or_resume_m4(
    PERSIST,
    M4_CONFIG,
    PROJECT_ROOT,
    run_id=M4_RUN_ID,
    test_report=M4_TEST_REPORT,
    allow_code_drift=True,
)
print('run_id', m4_orchestrator.state.run_id)
print('phase', m4_orchestrator.state.phase)
print('session_policy', m4_orchestrator.session_policy)
assert m4_orchestrator.state.certification_status == 'NOT_CERTIFIED'

M4_RESULT = m4_orchestrator.run_until_checkpoint()
assert M4_RESULT['certification_status'] == 'NOT_CERTIFIED'
print(json.dumps(M4_RESULT, indent=2, ensure_ascii=False, default=str))


## 進捗確認（何度でも可）


In [ ]:
from src.common import read_json

loaded = m4_orchestrator.checkpoints.load_latest(restore_rng=False)
if loaded is None:
    raise RuntimeError('No valid M4 checkpoint yet.')
inspection = {
    'run_id': loaded.state.run_id,
    'phase': loaded.state.phase,
    'checkpoint': str(loaded.path),
    'checkpoint_index': loaded.state.checkpoint_index,
    'done': sum(item.status == 'done' for item in loaded.queue.items.values()),
    'pending': sum(item.status == 'pending' for item in loaded.queue.items.values()),
    'failed': sum(item.status == 'failed' for item in loaded.queue.items.values()),
    'derivative_tensors': sorted(loaded.tensors),
    'certification_status': loaded.state.certification_status,
}
print(json.dumps(inspection, indent=2, ensure_ascii=False, default=str))

report_path = m4_orchestrator.run_root / 'reports' / 'M4_report.json'
if report_path.is_file():
    report = read_json(report_path)
    ledger = (report.get('results') or {}).get('M4_ERROR_LEDGER', {}).get('result', {})
    print(json.dumps({
        '判定': 'M4 report present (BLOCKED_MATH; NOT_CERTIFIED).',
        'phase': report.get('phase'),
        'milestone_status': report.get('milestone_status'),
        'acceptance_gates': report.get('acceptance_gates'),
        'missing_bound_terms': report.get('missing_bound_terms'),
        'partial_output_radii': ledger.get('aggregates'),
        'heuristic_results': report.get('heuristic_results'),
        'certification_status': report.get('certification_status'),
    }, indent=2, ensure_ascii=False, default=str))
else:
    print('M4 incomplete — re-run the session cell with the same M4_RUN_ID.')
